In [2]:
import os
os.chdir("..")

In [3]:
from transformers import AutoTokenizer
from datasets import DatasetDict,interleave_datasets,load_dataset

dataset=load_dataset(
    "json",
    data_files={
        'train':'data/instruction_train.jsonl',
        'validation':'data/instruction_validation.jsonl',
        'test':'data/instruction_test.jsonl',
    }
)

# Balance only the training sampler. Validation and test remain untouched.
summary_train=dataset['train'].filter(lambda ex:ex['task']=='policy_summary')
risk_train=dataset['train'].filter(lambda ex:ex['task']=='privacy_risk_extraction')

balanced_train=interleave_datasets(
    [summary_train,risk_train],
    probabilities=[0.5,0.5],
    seed=42,
    stopping_strategy='all_exhausted'
)
dataset=DatasetDict({
    'train':balanced_train,
    'validation':dataset['validation'],
    'test':dataset['test'],
})
print(dataset)
print(dataset['train'][0])


DatasetDict({
    train: Dataset({
        features: ['task', 'source_id', 'messages'],
        num_rows: 926
    })
    validation: Dataset({
        features: ['task', 'source_id', 'messages'],
        num_rows: 78
    })
    test: Dataset({
        features: ['task', 'source_id', 'messages'],
        num_rows: 78
    })
})
{'task': 'privacy_risk_extraction', 'source_id': 'opp:Amazon.json', 'messages': [{'role': 'user', 'content': 'TASK: PRIVACY_RISK_EXTRACTION\nReturn valid JSON with every required category. Use only verbatim evidence from the policy.\nUse [] when no evidence exists.\nCategories: BasicAccountInfo, ContactInfo, Demographic, GeoInfo, DeviceInfo, UsageData, InternetHistory, UserGenerated, UserProfileInfo, CommunicationProv, Payment, Financial, Purchase, HealthFitness, Biometric, Images, ContentPreferences, Settings, Metadata, Performance\n\nPOLICY:\n are data controllers of personal information collected and processed through Amazon Services. Details can be found here.

In [4]:
MODEL="Qwen/Qwen2.5-1.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(MODEL)

MAX_SEQUENCE_LENGTH=1024

def add_length(example):
    text=tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'length':len(tokenizer(text,add_special_tokens=False)['input_ids'])}

dataset=dataset.map(add_length)

for split in dataset:
    lengths=dataset[split]['length']
    print(split,'max:',max(lengths),'average:',sum(lengths)/len(lengths))


train max: 1024 average: 546.7969762419007
validation max: 1020 average: 473.0769230769231
test max: 1021 average: 501.8205128205128


In [5]:
long_summary = 0
long_analysis = 0


for ex in dataset['train']:
    if ex['length']>MAX_SEQUENCE_LENGTH:
        if ex['task']=='policy_summary':
            long_summary+=1
        else:
            long_analysis+=1

print("Long Summaries :", long_summary)
print("Long Analysis  :", long_analysis)

Long Summaries : 0
Long Analysis  : 0


In [6]:
filtered_dataset=dataset.filter(
    lambda example:example['length']<=MAX_SEQUENCE_LENGTH
)

In [7]:
filtered_dataset

DatasetDict({
    train: Dataset({
        features: ['task', 'source_id', 'messages', 'length'],
        num_rows: 926
    })
    validation: Dataset({
        features: ['task', 'source_id', 'messages', 'length'],
        num_rows: 78
    })
    test: Dataset({
        features: ['task', 'source_id', 'messages', 'length'],
        num_rows: 78
    })
})

In [8]:
import torch
from transformers import AutoModelForCausalLM,BitsAndBytesConfig

bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model=AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W0709 16:58:15.182000 2948 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

c:\Users\PratushPc\miniconda3\envs\policy_llm\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [8]:
from peft import LoraConfig

peft_config=LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    # Legacy value: lora_dropout=0.6,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="models/policy_assistant_v2",

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,

    logging_steps=10,

    fp16=True,

    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    optim="paged_adamw_8bit",

    weight_decay=0.01,

    report_to="none",

    remove_unused_columns=False,
)

In [ ]:
# sample = tokenizer.apply_chat_template(
#     dataset[0]["messages"],
#     tokenize=True,
#     return_dict=True,
#     return_assistant_tokens_mask=True
# )

# print(sample.keys())

In [ ]:
# batch = next(iter(trainer.get_train_dataloader()))

# labels = batch["labels"][0]

# print("Ignored tokens :", (labels == -100).sum().item())
# print("Training tokens:", (labels != -100).sum().item())

In [9]:
def preprocess(example):

    # Full conversation
    full_text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

    full = tokenizer(
        full_text,
        truncation=False,
        add_special_tokens=False,
    )
    

    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]

    # Conversation without assistant
    prompt_messages = example["messages"][:-1]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,   # leaves the template waiting for assistant
    )

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    assistant_start = len(prompt_ids)

    labels = input_ids.copy()

    labels[:assistant_start] = [-100] * assistant_start

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

In [10]:
processed_dataset=filtered_dataset.map(
    preprocess,
    remove_columns=filtered_dataset['train'].column_names
)

Map:   0%|          | 0/926 [00:00<?, ? examples/s]

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

In [11]:
processed_dataset.save_to_disk("data/final_ds")

Saving the dataset (0/1 shards):   0%|          | 0/926 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/78 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/78 [00:00<?, ? examples/s]

In [17]:
labels = processed_dataset['train'][0]["labels"]

ignored = labels.count(-100)

print("Ignored:", ignored)
print("Training:", len(labels) - ignored)

Ignored: 472
Training: 116


In [18]:
from peft import get_peft_model,prepare_model_for_kbit_training

model=prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)
model.config.use_cache=False
peft_model = get_peft_model(model, peft_config)

peft_model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [19]:
from transformers import DataCollatorForSeq2Seq,Trainer

data_collator=DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors='pt'
)

hf_trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=processed_dataset['train'],
    eval_dataset=processed_dataset['validation'],
    data_collator=data_collator
)

In [20]:
batch = next(iter(hf_trainer.get_train_dataloader()))

print(batch["input_ids"].shape)
print(batch["labels"].shape)

labels = batch["labels"][0]

print("Ignored:", (labels == -100).sum().item())
print("Training:", (labels != -100).sum().item())

torch.Size([1, 568])
torch.Size([1, 568])
Ignored: 472
Training: 96


In [21]:
train_result=hf_trainer.train(resume_from_checkpoint=None)
hf_trainer.save_model("models/policy_assistant_v2/best_adapter")
tokenizer.save_pretrained("models/policy_assistant_v2/best_adapter")

In [22]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

In [23]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [24]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [25]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

c:\Users\PratushPc\miniconda3\envs\policy_llm\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [ ]:
model = PeftModel.from_pretrained(
    base_model,
    "models/policy_assistant_v2/best_adapter"
)

In [27]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [15]:
ip=[filtered_dataset['test'][1]['messages'][0]]
ip

[{'role': 'user',
  'content': 'TASK: PRIVACY_RISK_EXTRACTION\nReturn valid JSON with every required category. Use only verbatim evidence from the policy.\nUse [] when no evidence exists.\nCategories: BasicAccountInfo, ContactInfo, Demographic, GeoInfo, DeviceInfo, UsageData, InternetHistory, UserGenerated, UserProfileInfo, CommunicationProv, Payment, Financial, Purchase, HealthFitness, Biometric, Images, ContentPreferences, Settings, Metadata, Performance\n\nPOLICY:\n the Platform. If you choose to find other users through your social network contacts, we will collect your public profile information as well as names and profiles of your social network contacts.\r\nPurchase Information. When you make a purchase or payment on or through the Platform, including when you buy TikTok Coins or purchase goods through our shopping features, we collect information about the purchase or payment transaction, such as payment card information, billing, delivery, and contact information, and items y

In [22]:
inputs = tokenizer.apply_chat_template(
    ip,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

inputs = inputs.to(model.device)

In [ ]:
generation_limit=(
    500 if ip[0]['content'].startswith('TASK: PRIVACY_RISK_EXTRACTION') else 220
)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=generation_limit,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

In [42]:
generated = outputs[0][inputs['input_ids'].shape[-1]:]

response = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print(response)

# Structured outputs must satisfy the complete schema and use verbatim evidence.
if ip[0]['content'].startswith('TASK: PRIVACY_RISK_EXTRACTION'):
    import json
    parsed=json.loads(response)
    expected_risk_keys={
        'BasicAccountInfo','ContactInfo','Demographic','GeoInfo','DeviceInfo',
        'UsageData','InternetHistory','UserGenerated','UserProfileInfo',
        'CommunicationProv','Payment','Financial','Purchase','HealthFitness',
        'Biometric','Images','ContentPreferences','Settings','Metadata','Performance'
    }
    missing=expected_risk_keys-set(parsed)
    print(f'Missing risk keys: {missing}')
    source=ip[0]['content'].split('POLICY:',1)[1]
    unsupported=[
        clause
        for kind in expected_risk_keys
        for clause in parsed[kind]
        if ' '.join(clause.split()).lower() not in ' '.join(source.split()).lower()
    ]

    print("clause not in policy : ",unsupported)

{"BasicAccountInfo": [], "Biometric": [], "CommunicationProv": [], "ContactInfo": ["contact information"], "ContentPreferences": [], "Demographic": [], "DeviceInfo": ["your device model", "operating system", "IP address", "system language", "device ID", "user ID"], "Financial": [], "GeoInfo": [], "HealthFitness": [], "Images": [], "InternetHistory": [], "Metadata": [], "Payment": ["payment card information"], "Performance": ["crash reports", "performance logs"], "Purchase": ["billing", "delivery", "contact information", "items you purchased"], "Settings": [], "UsageData": [], "UserGenerated": ["information about your use of the Platform", "feedback or inquiries"], "UserProfileInfo": []}
Missing risk keys: set()
clause not in policy :  ['information about your use of the Platform']
